In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data
from ts2vec.ts2vec import TS2Vec

In [2]:
seed = 1
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
input_dims = 6
output_dims = 32
hidden_dims = 64
depth = 6
batch_size = 32
max_train_length = 600
n_epochs = 300
patience = 15
patience_delta = 1e-3
lr = 0.001
n_splits = 5

In [3]:
# Parameters
seed = 6


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class EarlyStoppingException(Exception):
    pass

y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
scores = {k: np.full(len(y_true), np.nan) for k in ['lof', 'iso_forest', 'ocsvm']}

for fold, (train_idx, test_idx) in enumerate(skf.split(data_np_clean, y_true)):
    state = {'best_loss': float('inf'), 'patience_counter': 0}

    def after_epoch_callback(model, loss):
        if loss < state['best_loss'] - patience_delta:
            state['best_loss'] = loss
            state['patience_counter'] = 0
        else:
            state['patience_counter'] += 1
            if state['patience_counter'] >= patience:
                raise EarlyStoppingException()

    ts2vec = TS2Vec(
        input_dims=input_dims,
        output_dims=output_dims,
        hidden_dims=hidden_dims,
        depth=depth,
        device=device,
        lr=lr,
        batch_size=batch_size,
        max_train_length=max_train_length,
        after_epoch_callback=after_epoch_callback,
    )

    try:
        ts2vec.fit(data_np_clean[train_idx], n_epochs=n_epochs, verbose=False)
    except EarlyStoppingException:
        pass

    Z_train = ts2vec.encode(data_np_clean[train_idx], encoding_window='full_series')
    Z_test = ts2vec.encode(data_np_clean[test_idx], encoding_window='full_series')

    enc_scaler = StandardScaler()
    Z_train = enc_scaler.fit_transform(Z_train)
    Z_test = enc_scaler.transform(Z_test)

    lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
    lof.fit(Z_train)
    scores['lof'][test_idx] = -lof.score_samples(Z_test)

    iso = IsolationForest(random_state=rng.randint(1000))
    iso.fit(Z_train)
    scores['iso_forest'][test_idx] = -iso.score_samples(Z_test)

    ocsvm = OneClassSVM(kernel='rbf')
    ocsvm.fit(Z_train)
    scores['ocsvm'][test_idx] = -ocsvm.decision_function(Z_test)

    print(f"Fold {fold+1}/{n_splits} done")

Fold 1/5 done


Fold 2/5 done


Fold 3/5 done


Fold 4/5 done


Fold 5/5 done


In [9]:
for key, glue_name in [('lof', 'GBG500_ap_ts2vec_lof'),
                        ('iso_forest', 'GBG500_ap_ts2vec_iso_forest'),
                        ('ocsvm', 'GBG500_ap_ts2vec_ocsvm')]:
    ap = average_precision_score(y_true, scores[key])
    print(f"TS2Vec+{key} AP = {ap:.4f}")
    sb.glue(glue_name, float(ap))


TS2Vec+lof AP = 0.3604


TS2Vec+iso_forest AP = 0.0826


TS2Vec+ocsvm AP = 0.3565
